# 01 — Supervised Fine-Tuning (SFT)

Trains SFT model

## 1. Setup

In [ ]:
# Clone the repo (replace with your GitHub URL)
!git clone https://github.com/Roogard/math-rl-tuning.git
%cd math-rl-tuning

# Install the package in editable mode (`-e`), which means Python imports
# directly from the repo directory — so edits to the package take effect
# without reinstalling.
!pip install -e . --quiet
!pip install bitsandbytes --quiet  # QLoRA 4-bit quantization backend

## 2. Config

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA alloc conf: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'default')}")

from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, setup_wandb, mount_google_drive

# load_config() reads configs/default.yaml and returns a typed Config dataclass.
# Any value can be overridden programmatically (see next cell).
cfg = load_config()

# --- Authentication ---
# Option A: Set your tokens here
# setup_hf_token("hf_YOUR_TOKEN")
# setup_wandb(cfg.sft_training.wandb_project, key="YOUR_WANDB_KEY")

# Option B: Use Colab secrets (recommended — keeps tokens out of notebook history)
setup_hf_token()  # reads HF_TOKEN from Colab secrets or env var
setup_wandb(cfg.sft_training.wandb_project)

# Mount Google Drive so we can save the trained adapter and resume across sessions
mount_google_drive()

# --- Checkpoint Resume ---
# To resume from a crashed run, paste the checkpoint folder path here.
# Checkpoints are saved every 100 steps to outputs/sft/checkpoint-*/
# Leave as None to start fresh.
SFT_CHECKPOINT = None  # e.g. "/content/drive/MyDrive/math-rl-tuning/sft/checkpoint-100"

## 3. Prepare Data

In [ ]:
from math_rl_tuning.data import prepare_sft_data

# prepare_sft_data:
# 1. Loads NuminaMath-CoT (~860k examples) from Hugging Face
# 2. Injects the system prompt into each example's messages list
# 3. Normalizes all answers to \boxed{} format (converts GSM8K's "#### answer" style)
# 4. Filters out examples where no valid \boxed{} answer could be found
# 5. Keeps only the configured sources (gsm8k, math, cn_k12, etc.)
# 6. Balanced-samples up to train_per_source from each source (avoids large sources dominating)
train_ds, val_ds = prepare_sft_data(cfg)

print(f"\nTrain examples: {len(train_ds)}")
print(f"Val examples:   {len(val_ds)}")
print(f"\nSample (first message):")
print(train_ds[0]["messages"][0]["content"][:500])

## 4. Run SFT Training

In [ ]:
from math_rl_tuning.sft_trainer import run_sft_training

# run_sft_training:
# 1. Loads Qwen2.5-Math-7B with 4-bit NF4 quantization (QLoRA)
# 2. Attaches LoRA adapter (r=32, only ~1-2% of params are trainable)
# 3. Runs TRL's SFTTrainer with completion_only_loss=True
#    (only the assistant's response tokens contribute to the loss — not the question)
# 4. Saves the LoRA adapter to sft_output_dir (NOT the full merged model)
# 5. Optionally copies to Google Drive for persistence across Colab sessions
trainer, model, tokenizer = run_sft_training(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    save_to_drive=True,  # auto-copies to Google Drive
    checkpoint_path=SFT_CHECKPOINT,
)

## 5. Eval

In [7]:
from math_rl_tuning.inference import generate_stream

question = "Solve x + y = 10, 2x - y = 30."
print(f"Question: {question}\n")
response = generate_stream(question, model, tokenizer)

Question: Solve x + y = 10, 2x - y = 30.

To solve the system of equations x + y = 10 and 2x - y = 30, we can use the method of elimination. First, we can add the two equations together to eliminate y:

(x + y) + (2x - y) = 10 + 30
3x = 40

Next, we can solve for x by dividing both sides of the equation by 3:

x = 40 / 3
x = 20 / 3

Now that we have the value of x, we can substitute it back into one of the original equations to solve for y. Let's use the first equation:

x + y = 10
(20 / 3) + y = 10

To isolate y, we can subtract 20 / 3 from both sides of the equation:

y = 10 - (20 / 3)
y = (30 / 3) - (20 / 3)
y = 10 / 3

Therefore, the solution to the system of equations is x = 20 / 3 and y = 10 / 3.
The final answer is \boxed{x = \frac{20}{3}, y = \frac{10}{3}}.


## 6. Cleanup

In [8]:
from math_rl_tuning.utils import clean_memory

del model, trainer
clean_memory()

Memory cleared.
